In [ ]:
from Bio import SeqIO
import re

input_fasta = "../CZ/REPSEQ_RESULTS_3/phasmaviridae_species_30_2026_05_19/finalproteinsphasmaviridaerepseq3_hmm/Bunya_RdRp.fasta"
output_fasta = "../CZ/REPSEQ_RESULTS_3/phasmaviridae_species_30_2026_05_19/finalproteinsphasmaviridaerepseq3_hmm/Bunya_RdRp_sorted.fasta"
def extract_field(header, field):
    """
    Extract [field=value] from FASTA header.
    Returns None if field is missing.
    """
    match = re.search(rf"\[{field}=([^\]]+)\]", header)
    return match.group(1).strip().lower() if match else None

# Read sequences
records = list(SeqIO.parse(input_fasta, "fasta"))

# Flag missing annotations
for record in records:
    genus = extract_field(record.description, "genus")
    species = extract_field(record.description, "species")

    if genus is None or species is None:
        print(f"Missing annotation: {record.id}")
        print(record.description)
        print()

# Sort records
sorted_records = sorted(
    records,
    key=lambda r: (
        extract_field(r.description, "genus") or "",
        extract_field(r.description, "species") or ""
    )
)

# Write sorted FASTA
SeqIO.write(sorted_records, output_fasta, "fasta")

print(f"Sorted {len(sorted_records)} sequences.")
print(f"Output written to: {output_fasta}")

In [ ]:
#!/usr/bin/env python3
import argparse
import csv
import re
from pathlib import Path

import pandas as pd
from Bio import SeqIO


# ============================================
# Core Functions
# ============================================

def hamming_distance(s1, s2):
    """Calculate the Hamming distance between two strings."""
    if len(s1) != len(s2):
        raise ValueError("The strings must have the same length.")
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))


def extract_subgenus(header: str):
    """
    Extract genus and species from FASTA header.

    Example:
    [species=Altai virus]
    [genus=Orthohantavirus]

    Returns:
        "Orthohantavirus_Altai virus"
    """

    genus_match = re.search(r"\[genus=([^\]]+)\]", header)
    species_match = re.search(r"\[species=([^\]]+)\]", header)

    genus = genus_match.group(1).strip() if genus_match else ""
    species = species_match.group(1).strip() if species_match else ""

    if genus and species:
        return f"{genus}_{species}"
    elif genus:
        return genus
    elif species:
        return species
    else:
        return ""

def process_sequences(
    epitope_file,
    fasta_file,
    output_file,
    flank_size=None,
    epitope_seq_col="Sequence",
    start_col="Start",
    end_col="End",
):
    """
    Map epitopes to sequences and save results to Excel.

    flank_size=None → full search
    flank_size=int → windowed search ± flank_size aa
    """
    epitope_file = Path(epitope_file)
    fasta_file = Path(fasta_file)
    output_file = Path(output_file)

    # -------------------------
    # Load epitope CSV (robust header resolution)
    # -------------------------
    epitopes = []
    with epitope_file.open("r", newline="") as f:
        reader = csv.DictReader(f)
        raw_fields = reader.fieldnames or []
        norm_map = {name.strip().lower(): name for name in raw_fields}

        def resolve(col):
            key = col.strip().lower()
            if key in norm_map:
                return norm_map[key]
            raise KeyError(f"Column '{col}' not found. Available: {raw_fields}")

        seq_col_name = resolve(epitope_seq_col)
        start_col_name = resolve(start_col)
        end_col_name = resolve(end_col)

        for row in reader:
            epitopes.append({
                "seq": row[seq_col_name],
                "start": int(row[start_col_name]),
                "end": int(row[end_col_name]),
            })

    # -------------------------
    # Load FASTA
    # -------------------------
    sequences, headers, subgenera = [], [], []
    for record in SeqIO.parse(str(fasta_file), "fasta"):
        headers.append(record.id)
        sequences.append(str(record.seq))
        subgenera.append(extract_subgenus(record.description))

    # -------------------------
    # Output columns
    # -------------------------
    columns = ["epitope"]
    for h, sg in zip(headers, subgenera):
        columns += [
            f"{h}_{sg}_position",
            f"{h}_{sg}_score",
            f"{h}_{sg}_best_match",
        ]

    results_data = []
    multiple_best = []

    # -------------------------
    # Compute matching
    # -------------------------
    for epi in epitopes:
        epitope = epi["seq"]
        start_0 = epi["start"] - 1
        end_0 = epi["end"] - 1
        L = len(epitope)

        row = [epitope]

        for seq, h, sg in zip(sequences, headers, subgenera):
            seqlen = len(seq)

            if seqlen < L:
                row += [None, float("-inf"), None]
                continue

            if flank_size is None:
                # Full search
                w_start, w_end = 0, seqlen
            else:
                # Windowed
                w_start = max(0, start_0 - flank_size)
                w_end = min(seqlen, end_0 + flank_size + 1)

            if w_end - w_start < L:
                row += [None, float("-inf"), None]
                continue

            max_score = float("-inf")
            best_pos = None
            best_sub = None
            ties_pos = []
            ties_sub = []

            # sliding window
            for i in range(w_start, w_end - L + 1):
                subseq = seq[i:i+L]
                score = (L - hamming_distance(epitope, subseq)) / L

                if score > max_score:
                    max_score = score
                    best_pos = i
                    best_sub = subseq
                    ties_pos = [i]
                    ties_sub = [subseq]
                elif score == max_score:
                    ties_pos.append(i)
                    ties_sub.append(subseq)

            row += [best_pos, max_score, best_sub]

            if len(ties_pos) > 1:
                multiple_best.append({
                    "epitope": epitope,
                    "header": h,
                    "subgenus": sg,
                    "max_score": max_score,
                    "all_positions": ";".join(map(str, ties_pos)),
                    "all_matches": ";".join(ties_sub),
                    "chosen_position": best_pos,
                    "chosen_match": best_sub,
                    "flank_size": flank_size if flank_size is not None else "full",
                })

        results_data.append(row)

    # -------------------------
    # Save Excel
    # -------------------------
    df = pd.DataFrame(results_data, columns=columns)

# ✅ Convert main *_position columns to 1-based
    for col in df.columns:
        if col.endswith("_position"):
            df[col] = df[col].apply(lambda x: x + 1 if pd.notna(x) else x)

# ✅ Convert Multiple_Best_Matches positions to 1-based
    for rec in multiple_best:
    # chosen_position: int or None
        if rec["chosen_position"] is not None:
            rec["chosen_position"] += 1

    # all_positions: string like "0;5;10"
        if rec["all_positions"]:
            rec["all_positions"] = ";".join(
                str(int(p) + 1) for p in rec["all_positions"].split(";")
        )

    df_scores = df[["epitope"] + [c for c in df.columns if c.endswith("_score")]]

    if multiple_best:
        df_ties = pd.DataFrame(multiple_best)
    else:
        df_ties = pd.DataFrame(columns=[
            "epitope","header","subgenus","max_score",
            "all_positions","all_matches",
            "chosen_position","chosen_match","flank_size"
    ])

    with pd.ExcelWriter(output_file) as w:
        df.to_excel(w, sheet_name="Original", index=False)
        df_scores.to_excel(w, sheet_name="All_Scores", index=False)
        df_ties.to_excel(w, sheet_name="Multiple_Best_Matches", index=False)


# ============================================
# Configuration: EDIT YOUR PATHS HERE
# ============================================
THRESHOLDS = [1.0, 0.9, 0.8, 0.7, 0.5]

proteins = {
     "RdRp": {
             "epitopes": "../RNApolymerase_15mers.csv",
         "fasta": "../CZ/REPSEQ_RESULTS_3/peribunyaviridae_species_30_2026_05_19/finalproteinsperibunyaviridaerepseq3_hmm/merged/rdrp_sorted.fasta",
         "flanks": [25,50,100,200],
      },
       "GnGc": {
          "epitopes": "../GnGc_15mers.csv",
          "fasta": "../CZ/REPSEQ_RESULTS_3/peribunyaviridae_species_30_2026_05_19/finalproteinsperibunyaviridaerepseq3_hmm/merged/glycoprotein_sorted.fasta",
          "flanks": [25,50,100,200],
      },
     
     "Nucleocapsid": {
         "epitopes": "../Nucleocapsid_15mers.csv",
         "fasta": "../CZ/REPSEQ_RESULTS_3/peribunyaviridae_species_30_2026_05_19/finalproteinsperibunyaviridaerepseq3_hmm/merged/nucleoprotein_sorted.fasta",
         "flanks": [25,50,100,200],
     },
      "Nonstructural": {
          "epitopes": "../Nonstructural_15mers.csv",
          "fasta": "../CZ/REPSEQ_RESULTS_3/peribunyaviridae_species_30_2026_05_19/finalproteinsperibunyaviridaerepseq3_hmm/merged/nonstructural_sorted.fasta",
          "flanks": [25,50,100,200],
      }
}

RESULTS_DIR = Path("conservation_results_peribunyaviridae_repseq3_species_cz")
RESULTS_DIR.mkdir(exist_ok=True)


# ============================================
# Batch Runners
# ============================================

def run_mappings():
    """Run full and flank searches."""
    for protein, cfg in proteins.items():
        epi = cfg["epitopes"]
        fas = cfg["fasta"]

        out_full = RESULTS_DIR / f"{protein}_fullsearch.xlsx"
        print(f"[{protein}] FULL search → {out_full}")
        process_sequences(epitope_file=epi, fasta_file=fas, output_file=out_full, flank_size=None)

        for flank in cfg["flanks"]:
            out_flank = RESULTS_DIR / f"{protein}_flank{flank}.xlsx"
            print(f"[{protein}] flank={flank} → {out_flank}")
            process_sequences(epitope_file=epi, fasta_file=fas, output_file=out_flank, flank_size=flank)


def run_comparisons():
    """
    Compare flank-limited runs vs full search for each protein,
    for multiple score thresholds.

    For each threshold t, we measure:

        fraction(t) = (# pairs with s_full >= t AND same position)
                       --------------------------------------------
                       (# pairs with s_full >= t)
    """
    all_rows = []

    for protein, cfg in proteins.items():
        full_file = RESULTS_DIR / f"{protein}_fullsearch.xlsx"
        full = pd.read_excel(full_file, sheet_name="Original").set_index("epitope")

        pos_full = [c for c in full.columns if c.endswith("_position")]

        print(f"\n=== Comparing flank sizes for {protein} ===")

        for flank in cfg["flanks"]:
            flank_file = RESULTS_DIR / f"{protein}_flank{flank}.xlsx"
            df_flank = pd.read_excel(flank_file, sheet_name="Original").set_index("epitope")
            pos_flank = [c for c in df_flank.columns if c.endswith("_position")]

            common = sorted(set(pos_full) & set(pos_flank))
            if not common:
                print(f"  [WARN] No common columns for flank={flank}")
                continue

            # Loop over thresholds
            for t in THRESHOLDS:
                total_t = 0
                matched_t = 0

                for col in common:
                    col_score_full = col.replace("_position", "_score")
                    col_score_flank = col.replace("_position", "_score")

                    p_full = full[col]
                    p_flank = df_flank[col]
                    s_full = full[col_score_full]

                    mask = ~p_full.isna()
                    threshold_mask = mask & (s_full >= t)

                    total_t += threshold_mask.sum()
                    matched_t += (threshold_mask & (p_full == p_flank)).sum()

                fraction_t = matched_t / total_t if total_t else None

                print(f"  flank={flank}, threshold={t}: fraction={fraction_t}")

                all_rows.append({
                    "protein": protein,
                    "flank_size": flank,
                    "threshold": t,
                    "total_pairs": total_t,
                    "matched_pairs": matched_t,
                    "fraction": fraction_t,
                })

    summary = pd.DataFrame(all_rows)
    summary.to_csv(RESULTS_DIR / "flank_threshold_summary.csv", index=False)
    print("\nSaved results to flank_threshold_summary.csv")

    

# ============================================
# JUPYTER-FRIENDLY MAIN FUNCTION
# ============================================

def main(action="run-all"):
    """
    Default action = run-all (safe for Jupyter).
    Ignoring Jupyter's extra CLI args.
    """
    if action == "run-mappings":
        run_mappings()
    elif action == "run-comparisons":
        run_comparisons()
    else:
        run_mappings()
        run_comparisons()


# ============================================
# END OF SCRIPT
# ============================================
main()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re
from pathlib import Path


def protein_name_from_filename(filename: str) -> str:
    name = filename.lower()

    if name.startswith("rdrp_"):
        return "RdRp Protein"
    elif name.startswith("nonstructural_"):
        return "NS Protein"
    elif name.startswith("nucleocapsid_"):
        return "N Protein"
    elif name.startswith("gngc_"):
        return "GnGc Protein"
    else:
        return "Protein"

def extract_species_genus(col):
    """
    Extract species and genus from:
    QOI08666.1_Orthohantavirus_Altai virus_score
    """
    name = str(col).removesuffix("_score")
    parts = name.split("_")

    if len(parts) >= 3:
        genus = parts[-2]
        species = parts[-1]
        return species, genus

    return "Unknown", "Unknown"

input_dir = Path("conservation_results_hantavirus_repseq3_species_cz2/modified_nan_noinfseqs/")
excel_files = sorted(input_dir.glob("*.xlsx"))

print(f"Found {len(excel_files)} files")

for excel_file in excel_files:
    results_df = pd.read_excel(excel_file, sheet_name="All_Scores")

    score_columns = [c for c in results_df.columns if str(c).endswith("_score")]
    if not score_columns:
        print(f"Skipping {excel_file.name}: no score columns")
        continue

    heatmap_data = results_df[score_columns].copy()

    mask_empty = results_df["epitope"].isna() | (
        results_df["epitope"].astype(str).str.strip() == ""
    )
    heatmap_data.loc[mask_empty, :] = np.nan

    species_genus = [extract_species_genus(c) for c in score_columns]
    species = [x[0] for x in species_genus]
    genera = [x[1] for x in species_genus]

    # Preserve Excel column order
    score_columns_sorted = score_columns
    species_sorted = species
    genera_sorted = genera

    heatmap_data = heatmap_data[score_columns_sorted]

    heatmap_matrix = heatmap_data.to_numpy(dtype=float)
    mask = np.isnan(heatmap_matrix)
    nrows, ncols = heatmap_matrix.shape

    # Species separator boundaries
    species_change_points = [
        i for i in range(1, ncols)
        if species_sorted[i] != species_sorted[i - 1]
    ]
    species_boundaries = [0] + species_change_points + [ncols]

    # Genus label centers
    genus_change_points = [
        i for i in range(1, ncols)
        if genera_sorted[i] != genera_sorted[i - 1]
    ]
    genus_boundaries = [0] + genus_change_points + [ncols]

    genus_names = []
    genus_centers = []

    for start, end in zip(genus_boundaries[:-1], genus_boundaries[1:]):
        genus_names.append(genera_sorted[start])
        genus_centers.append((start + end) / 2.0)

    epitope_numbers = [str(i) for i in range(1, nrows + 1)]

    ystep = 1 if nrows <= 50 else (2 if nrows <= 100 else 5)

    plt.figure(figsize=(18, 18))

    ax = sns.heatmap(
        heatmap_matrix,
        mask=mask,
        cmap="RdBu",
        vmin=0,
        vmax=1,
        cbar_kws={"label": "Similarity Score"},
        linewidths=0
    )

    sns.heatmap(
        np.zeros_like(heatmap_matrix),
        mask=~mask,
        cmap=["#4d4d4d"],
        cbar=False,
        ax=ax
    )

    # Species separator lines
    for b in species_boundaries[1:-1]:
        ax.axvline(b, color="black", linewidth=0.35)

    # Hide individual sequence x labels
    ax.set_xticks([])
    ax.set_xticklabels([])

    # Genus labels on x axis
    secax = ax.secondary_xaxis("bottom")
    secax.set_xticks(genus_centers)
    secax.set_xticklabels(
        genus_names,
        rotation=90,
        ha="center",
        va="top",
        fontsize=6,
        fontweight="bold"
    )
    secax.spines["bottom"].set_visible(False)
    secax.tick_params(axis="x", length=0, pad=20)
    secax.set_xlabel("Genus", labelpad=35)

    # Y-axis peptide labels
    yticks = np.arange(0, nrows, ystep) + 0.5
    ax.set_yticks(yticks)
    ax.set_yticklabels(
        [epitope_numbers[i] for i in range(0, nrows, ystep)],
        fontsize=9
    )

    ax.set_ylabel("Peptide Index", fontsize=12)

    title_name = excel_file.stem.replace("flank", "window")
    protein_name = protein_name_from_filename(excel_file.name)

    ax.set_title(
        f"{protein_name} Peptide Similarity Grouped by Genus\n{title_name}",
        fontsize=14,
        pad=20
    )

    plt.subplots_adjust(bottom=0.42)

    output_svg = excel_file.with_suffix(".svg")
    plt.savefig(output_svg, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Saved {output_svg}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Load your threshold summary
df = pd.read_csv("conservation_results_peribunyaviridae_repseq3_species_cz/flank_threshold_summary.csv")

# Make a folder for plots
os.makedirs("plots", exist_ok=True)

proteins = df["protein"].unique()

for prot in proteins:
    sub = df[df["protein"] == prot]

    # Create figure and axes
    fig, ax = plt.subplots(figsize=(8, 5))

    # Plot each threshold separately
    for t in sorted(sub["threshold"].unique()):
        sub_t = sub[sub["threshold"] == t].sort_values("flank_size")
        ax.plot(sub_t["flank_size"], sub_t["fraction"], marker="o", label=f"t ≥ {t}")

    ax.set_title(f"{prot}: Recovery vs Window size (all thresholds)")
    ax.set_xlabel("Window size (aa)")
    ax.set_ylabel("Recovery fraction")
    ax.set_ylim(0, 1.05)
    ax.grid(True)
    ax.legend()

    # Save PNG file *before* closing
    outname = f"plots/{prot}_window_threshold_plot.svg"
    fig.savefig(outname, dpi=300, bbox_inches="tight")
    print(f"Saved: {outname}")

    # Now show in notebook (optional)
    plt.show()
    plt.close(fig) 

